In [0]:
from pyspark.sql import SparkSession
from datetime import datetime 
import requests
import zipfile
import pyspark.sql.functions as F
import os
import glob
import shutil
import pandas as pd



spark.sql("CREATE VOLUME IF NOT EXISTS cnpjs_schema.enterprises")
spark.sql("CREATE VOLUME IF NOT EXISTS cnpjs_schema.establishments")
spark.sql("CREATE VOLUME IF NOT EXISTS cnpjs_schema.tax_regime_data")
spark.sql("CREATE VOLUME IF NOT EXISTS cnpjs_schema.auxiliar")
dbutils.fs.mkdirs('/Volumes/workspace/cnpjs_schema/tax_regime_data/raw')
dbutils.fs.mkdirs('/Volumes/workspace/cnpjs_schema/tax_regime_data/bronze')
dbutils.fs.mkdirs('/Volumes/workspace/cnpjs_schema/establishments/bronze')
dbutils.fs.mkdirs('/Volumes/workspace/cnpjs_schema/auxiliar/legal_natures')
dbutils.fs.mkdirs('/Volumes/workspace/cnpjs_schema/auxiliar/CNAE/raw')


In [0]:
## ---------- Creating url_date ----------
def get_month():
    today = datetime.now()
    actual_month = today.month 
    actual_year = today.year

    if today.month == 1:

        actual_month = 12
        actual_year = today.year - 1
        url_date = f'{actual_year}-{actual_month}'
    else:
        actual_month = today.month - 1
        url_date = f'{actual_year}-{actual_month:02d}'
        return(url_date)

url_date = get_month()


In [0]:
## ------------------- Getting CNAE Codes -----------------------
url_date = get_month()
cnaes_path = f'https://arquivos.receitafederal.gov.br/public.php/dav/files/gn672Ad4CF8N6TK/Dados/Cadastros/CNPJ/{url_date}/Cnaes.zip'
cnaes_raw_path = f'/Volumes/workspace/cnpjs_schema/auxiliar/CNAE/raw/{url_date}_Cnaes.zip'

r = requests.get(cnaes_path)
r.raise_for_status()

with open (cnaes_raw_path, 'wb') as f:
    f.write(r.content)

with zipfile.ZipFile(cnaes_raw_path, 'r') as z:
    original_name = z.namelist()[0]
    z.extract(original_name, '/Volumes/workspace/cnpjs_schema/auxiliar/CNAE/')
    os.rename(f'/Volumes/workspace/cnpjs_schema/auxiliar/CNAE/{original_name}' , f'/Volumes/workspace/cnpjs_schema/auxiliar/CNAE/{url_date}_Cnaes.csv')




In [0]:
## ------------ Getting Legal Nature Codes ----------------
url = f'https://arquivos.receitafederal.gov.br/public.php/dav/files/gn672Ad4CF8N6TK/Dados/Cadastros/CNPJ/{url_date}/Naturezas.zip'

r = requests.get(url)
r.raise_for_status()


with open (f'/Volumes/workspace/cnpjs_schema/auxiliar/legal_natures/{url_date}_Naturezas.zip', 'wb') as f:
    f.write(r.content)



In [0]:
## ------------- Extracting Legal Nature Codes -------------
with zipfile.ZipFile(f'/Volumes/workspace/cnpjs_schema/auxiliar/legal_natures/{url_date}_Naturezas.zip', 'r') as z:
    original_name = z.namelist()[0]
    print(original_name)
    z.extract(original_name, '/Volumes/workspace/cnpjs_schema/auxiliar/legal_natures/')
    os.rename(f'/Volumes/workspace/cnpjs_schema/auxiliar/legal_natures/{original_name}' , f'/Volumes/workspace/cnpjs_schema/auxiliar/legal_natures/{url_date}_Naturezas.csv')



In [0]:

# -------Getting tax_regime files from Receita Federal website-------

# urls_dict = {
#     'imunes_url':'https://arquivos.receitafederal.gov.br/dados/cnpj/regime_tributario/Imunes%20e%20Isentas.zip',
#     'lucro_arbitrado':'https://arquivos.receitafederal.gov.br/dados/cnpj/regime_tributario/Lucro%20Arbitrado.zip',
#     'lucro_real':'https://arquivos.receitafederal.gov.br/dados/cnpj/regime_tributario/Lucro%20Real.zip',
#     'lucro_presumido':'https://arquivos.receitafederal.gov.br/dados/cnpj/regime_tributario/Lucro%20Presumido.zip'
# }

# for url in urls_dict.items():
#     r = requests.get(url[1])
#     r.raise_for_status()
#     archive_name = url[0]
#     with open (f'/Volumes/workspace/cnpjs_schema/tax_regime_data/raw/{archive_name}.zip', 'wb') as f:
#        f.write(r.content)

In [0]:
## ----------- Extracting All Tax Regime files ---------------
# path = '/Volumes/workspace/cnpjs_schema/tax_regime_data/raw'
# files_path = os.listdir('/Volumes/workspace/cnpjs_schema/tax_regime_data/raw')

# for file in files_path:
#     archive = os.path.join(path, file)
#     with zipfile.ZipFile(archive, 'r') as zipped:
#         zipped.extractall(f'/Volumes/workspace/cnpjs_schema/tax_regime_data/bronze/')




In [0]:
## -------Getting enterprises files from Receita Federal website-------
dbutils.fs.mkdirs(f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/raw/')
dbutils.fs.mkdirs(f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/bronze/')

try:
    enterprises_archives = os.listdir(f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/raw/')
except:
    print('No files on directory')

for i in range(10):
    enterprise_name_for_read = f'{url_date}_enterprise{i}.zip'

    if enterprise_name_for_read in enterprises_archives:
        print(f'Archive {enterprise_name_for_read} already exists, reading the next archive...') 
        continue
    
    else:
        print(f'Archive {enterprise_name_for_read} not found. Going to download...')

    try:
        print(f'Starting download {url_date}_enterprises_{i}')
        r = requests.get(f'https://arquivos.receitafederal.gov.br/public.php/dav/files/gn672Ad4CF8N6TK/Dados/Cadastros/CNPJ/{url_date}/Empresas{i}.zip'
            )
        r.raise_for_status()
        with open(f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/raw/{url_date}_enterprise{i}.zip', 
                  mode='wb') as w:            
            w.write(r.content)
        print (f'{url_date}enterprise{i}.zip archive has downloaded')

    except requests.exceptions.RequestException as e:
        
        print(f'error on download {url_date}enterprise{i}.zip: {e}')
        break


In [0]:
## ----------- Extracting All Enterprise files ---------------
path = f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/raw/'
files_path = glob.glob(f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/raw/{url_date}*.zip')

print(f'Files to extract: {files_path}')

for archive in files_path:
    with zipfile.ZipFile(archive, 'r') as zipped:
        original_name = zipped.namelist()[0]
        zipped.extract(original_name,f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/bronze/')
        shutil.move(
            f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/bronze/{original_name}', 
            f'/Volumes/workspace/cnpjs_schema/enterprises/{url_date}_data/bronze/{url_date}_{original_name}'
            )
        print (f'{url_date}_{original_name} has been extracted')


In [0]:
## --------- Getting all establishments files ------------------
dbutils.fs.mkdirs(f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/raw/')
dbutils.fs.mkdirs(f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/bronze/')

establishments_archives = os.listdir(f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/raw/')

for i in range(10):
    establishments_name_for_read = f'{url_date}_establishment{i}.zip'

    if establishments_name_for_read in establishments_archives:
        print(f'Archive {establishments_name_for_read} already exists, reading the next archive...')
        
        continue
    
    else:
        print(f'Archive {establishments_name_for_read} not found. Going to download...')

    try:
        print(f'Starting download {url_date}_establishments_{i}')
        r = requests.get(f'https://arquivos.receitafederal.gov.br/public.php/dav/files/gn672Ad4CF8N6TK/Dados/Cadastros/CNPJ/{url_date}/Estabelecimentos{i}.zip')
        r.raise_for_status()
        with open(f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/raw/{url_date}_establishment{i}.zip', mode='wb') as w:
            w.write(r.content)
        print (f'{url_date}estabblishment{i}.zip archive has downloaded')
    except requests.exceptions.RequestException as e:
        print(f'error on download establishment{url_date}{i}: {e}')
        break

In [0]:
## ----------- Extracting establishements files ---------------

files_path = glob.glob(f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/raw/{url_date}*.zip')
print(f'files to extract: {files_path}')

for archive in files_path:
    with zipfile.ZipFile(archive, 'r') as zipped:
        original_name = zipped.namelist()[0]
        print(f'original name:')
        zipped.extract(original_name, f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/bronze/')
        shutil.move(
            f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/bronze/{original_name}', 
            f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/bronze/{url_date}_{original_name}'
            )
        print (f'{url_date}_{original_name} has been extracted')
